## GradCAM++ Heatmap Visualization

GradCAM++ interpretability for ResGAT. Representative case: `S22`.

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import h5py
import cv2

from models import ResGATs

%matplotlib inline
plt.rcParams['figure.dpi'] = 150

### Configuration

In [ ]:
GRAPH_DIR     = "graph_construction/graphs/"
CKPT_DIR      = "models/weights/"
FEATURES_DIR  = ""                                     # directory containing h5_files/
WSI_DIR       = ""                                     # directory containing .svs files

SLIDE_NAME    = "S22"
TARGET_LAYER  = "block3"
TARGET_CLASS  = None                                   # None = use true label
DEVICE        = "cuda:0" if torch.cuda.is_available() else "cpu"

In [ ]:
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, f"Weighted_Graph_{SLIDE_NAME}*")))
assert len(graph_files) > 0, f"No graph found for {SLIDE_NAME} in {GRAPH_DIR}"
graph_path = graph_files[0]
print(f"Graph: {graph_path}")

graph = torch.load(graph_path, map_location="cpu", weights_only=False)
true_label = int(graph.y.item()) if hasattr(graph, 'y') else 0
in_dim = graph.x.size(1)
print(f"Nodes: {graph.x.size(0)}, Features: {in_dim}, Label: {true_label}")

ckpt_paths = sorted(glob.glob(os.path.join(CKPT_DIR, "ResGAT_fold*_best.pth")))
print(f"Found {len(ckpt_paths)} fold checkpoints")

### Computation Method

In [ ]:
def explain_with_gradcam_pp(model, graph, target_class, device,
                            target_layer="block3"):
    """Compute per-node GradCAM++ importance scores."""
    model.eval()
    activations = {}

    def save_activation(name):
        def hook(module, inp, out):
            activations[name] = out
            out.retain_grad()
        return hook

    block = getattr(model, target_layer)
    handle = block.register_forward_hook(save_activation("target"))

    data = graph.to(device)
    if not hasattr(data, 'batch') or data.batch is None:
        data.batch = torch.zeros(data.x.size(0), dtype=torch.long,
                                 device=device)

    logits, _ = model(data)
    score = logits[0, target_class]

    model.zero_grad()
    score.backward()

    target_act = activations["target"]
    target_grad = target_act.grad

    # GradCAM++ alpha weights
    grad_2 = target_grad.pow(2)
    grad_3 = target_grad.pow(3)
    sum_grad_2 = grad_2.sum(dim=0, keepdim=True)
    sum_grad_3 = grad_3.sum(dim=0, keepdim=True)
    alpha = sum_grad_2 / (2 * sum_grad_2 +
                          sum_grad_3 * target_act.shape[0] + 1e-8)

    weights = alpha * F.relu(target_grad)
    cam_weights = weights.mean(dim=0, keepdim=True)
    cam = (cam_weights * target_act).sum(dim=1)
    cam = F.relu(cam)

    handle.remove()

    node_imp = cam.detach().cpu().numpy()
    if node_imp.max() > node_imp.min():
        node_imp = (node_imp - node_imp.min()) / (node_imp.max() - node_imp.min())
    else:
        node_imp = np.ones_like(node_imp) * 0.5
    return node_imp

In [ ]:
def load_coords_and_thumbnail(slide_name, features_dir, wsi_dir,
                               max_thumb=(5000, 5000)):
    """Load patch coordinates from h5 and WSI thumbnail."""
    import openslide

    h5_dir = os.path.join(features_dir, "h5_files")
    h5_hits = sorted(glob.glob(os.path.join(h5_dir, f"{slide_name}*.h5")))
    assert h5_hits, f"No h5 file found for {slide_name}"
    h5_path = h5_hits[0]

    with h5py.File(h5_path, "r") as f:
        coords = f["coords"][:].astype(np.float32)
        attrs = {k: v for k, v in f["coords"].attrs.items()}

    ld = np.array(attrs.get("level_dim", ()))
    dd = np.array(attrs.get("downsampled_level_dim", ()))
    if ld.size and dd.size and float(dd[0]) != 0:
        scale_corr = int(round(float(ld[0]) / float(dd[0])))
        coords = coords * scale_corr

    patch_size = int(attrs.get("patch_size", 256))

    svs_hits = sorted(glob.glob(os.path.join(wsi_dir, f"*{slide_name}*.svs")))
    assert svs_hits, f"No .svs file found for {slide_name}"
    slide = openslide.OpenSlide(svs_hits[0])
    w_orig, h_orig = slide.dimensions
    thumbnail = slide.get_thumbnail(max_thumb)
    thumb_w, thumb_h = thumbnail.size
    scale_x, scale_y = thumb_w / w_orig, thumb_h / h_orig
    coords_scaled = coords * np.array([scale_x, scale_y])

    patch_w = int(max(1, round(patch_size * scale_x)))
    patch_h = int(max(1, round(patch_size * scale_y)))
    side = max(patch_w, patch_h)

    return thumbnail, coords_scaled, (thumb_h, thumb_w), (side, side)


def paint_heatmap(coords_scaled, scores, out_hw, patch_hw):
    """Paint per-patch scores into a dense thumbnail-space heatmap."""
    H, W = out_hw
    pw, ph = patch_hw
    m = np.zeros((H, W), dtype=np.float32)
    xs = np.floor(coords_scaled[:, 0]).astype(int)
    ys = np.floor(coords_scaled[:, 1]).astype(int)
    for i in range(min(len(xs), len(scores))):
        x0, y0 = max(0, xs[i]), max(0, ys[i])
        x1 = min(W, x0 + pw + 1)
        y1 = min(H, y0 + ph + 1)
        if scores[i] > 0:
            m[y0:y1, x0:x1] = np.maximum(m[y0:y1, x0:x1], scores[i])
    return m


def extract_roi_contours(heatmap, threshold=0.5, min_area=200):
    """Extract ROI contours from a normalised heatmap."""
    mask = (heatmap > threshold).astype(np.uint8) * 255
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    contours = [c for c in contours if cv2.contourArea(c) >= min_area]
    return contours


def overlay_heatmap(thumbnail, heatmap, alpha=0.35, cmap="jet"):
    """Blend a heatmap onto a thumbnail image."""
    base = np.asarray(thumbnail, dtype=np.float32) / 255.0
    h_min, h_max = heatmap.min(), heatmap.max()
    if h_max - h_min > 1e-7:
        norm = (heatmap - h_min) / (h_max - h_min)
    else:
        norm = np.zeros_like(heatmap)
    cmap_fn = plt.cm.get_cmap(cmap)
    colored = cmap_fn(norm)[..., :3]
    blended = np.clip((1 - alpha) * base + alpha * colored, 0, 1)
    return blended, norm

### Per-Fold GradCAM++

In [ ]:
device = torch.device(DEVICE)
target_class = TARGET_CLASS if TARGET_CLASS is not None else true_label

model = ResGATs(in_dim).to(device)

per_fold_imps = []
for ckpt_path in ckpt_paths:
    fold_tag = os.path.splitext(os.path.basename(ckpt_path))[0]
    print(f"Loading {fold_tag} ...")
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]
    model.load_state_dict(state, strict=False)
    model.to(device)

    imp = explain_with_gradcam_pp(model, graph, target_class, device,
                                  target_layer=TARGET_LAYER)
    per_fold_imps.append(imp)
    print(f"  {fold_tag}: mean={imp.mean():.4f}, max={imp.max():.4f}")

print(f"\nCollected {len(per_fold_imps)} folds")

### Aggregated Heatmap

In [ ]:
def weighted_fuse(imps_list, weights=None):
    """Fuse per-fold importance maps with min-max normalisation."""
    normed = []
    for a in imps_list:
        a = a.astype(np.float32).ravel()
        mn, mx = a.min(), a.max()
        normed.append((a - mn) / (mx - mn + 1e-12))
    X = np.stack(normed, axis=0)
    if weights is None:
        weights = np.ones(len(imps_list), dtype=np.float32)
    w = np.array(weights, dtype=np.float32)
    w = w / w.sum()
    fused = (w[:, None] * X).sum(axis=0)
    return fused

aggregated_imp = weighted_fuse(per_fold_imps)
print(f"Aggregated: mean={aggregated_imp.mean():.4f}, max={aggregated_imp.max():.4f}")

### Combined Figure

In [ ]:
thumbnail, coords_scaled, out_hw, patch_hw = load_coords_and_thumbnail(
    SLIDE_NAME, FEATURES_DIR, WSI_DIR)

# Panel A: Original WSI + ROI
agg_hmap = paint_heatmap(coords_scaled, aggregated_imp, out_hw, patch_hw)
contours = extract_roi_contours(agg_hmap, threshold=0.5)

original_rgb = np.asarray(thumbnail, dtype=np.uint8).copy()
cv2.drawContours(original_rgb, contours, -1, (255, 0, 0), 3)

# Panel B: Aggregated heatmap overlay
agg_overlay, agg_norm = overlay_heatmap(thumbnail, agg_hmap)

# Panels C-G: Per-fold heatmaps
fold_overlays = []
for imp in per_fold_imps:
    hmap = paint_heatmap(coords_scaled, imp, out_hw, patch_hw)
    ov, _ = overlay_heatmap(thumbnail, hmap)
    fold_overlays.append(ov)

In [ ]:
n_folds = len(fold_overlays)
n_cols = 2 + n_folds

fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))

axes[0].imshow(original_rgb)
axes[0].set_title(f"{SLIDE_NAME} — Original + ROI", fontsize=10)
axes[0].axis("off")

axes[1].imshow(agg_overlay)
axes[1].set_title("Aggregated GradCAM++", fontsize=10)
axes[1].axis("off")

cax = inset_axes(axes[1], width="3%", height="50%", loc="lower right",
                 bbox_to_anchor=(0.05, 0, 1, 1),
                 bbox_transform=axes[1].transAxes, borderpad=0)
sm = plt.cm.ScalarMappable(norm=mcolors.Normalize(0, 1),
                            cmap=plt.cm.get_cmap("jet"))
sm.set_array([])
fig.colorbar(sm, cax=cax)

for i, ov in enumerate(fold_overlays):
    axes[2 + i].imshow(ov)
    axes[2 + i].set_title(f"Fold {i + 1}", fontsize=10)
    axes[2 + i].axis("off")

fig.suptitle(f"GradCAM++ — {SLIDE_NAME}", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"heatmap_{SLIDE_NAME}_combined.png", dpi=300,
            bbox_inches="tight")
plt.show()